- verl version
    - 0.7.0.dev0
    - 如果大家手头是更新的版本，也可以梳理下改进了什么，重构了什么

### scripts

- retool
    - https://github.com/verl-project/verl-recipe/blob/3490a22a0a3adeb7e4787fe70b1060b642efbae4/retool/run_qwen2_7b_dapo.sh
- https://github.com/verl-project/verl/blob/main/examples/sglang_multiturn/run_qwen3_4b_dapo_multiturn.sh
- parameters & config
    - `actor_rollout_ref.rollout.mode=async \` => AgentLoopManager
    - `actor_rollout_ref.rollout.multi_turn.format=hermes \`

```sh
tool_config_path=recipe/retool/sandbox_fusion_tool_config.yaml

actor_rollout_ref.rollout.multi_turn.tool_config_path=$tool_config_path 
```

- `sandbox_fusion_tool_config.yaml`
    - `sandbox_fusion_url: "http://localhost:8080/run_code"`
    - num_workers & rate_limit

### verl 中的 tool 定义与实现

- retool.py: `class CustomSandboxFusionTool(SandboxFusionTool):`
    - `class SandboxFusionTool(BaseTool):`
        - create
        - execute
            - `self.execution_pool.execute.remote(self.execute_code, instance_id, code, timeout, language)`
    - CustomSandboxFusionTool 主要是重写了 execute 方法
        - 提取 Markdown 代码块 (预处理 1)
        - 强制打印最后一行结果 (预处理 2)
            - 因为后端的沙箱执行服务依赖 stdout（标准输出）来获取结果，如果大模型生成的代码只做了计算而没有 print，沙箱将返回空结果。所以这里它逆序遍历代码行，找到最后一行非空行，如果该行不是以 print 开头，就强制给它套上 print(...)。

### Env (sandbox)

- `docker run -it -p 8080:8080 volcengine/sandbox-fusion:server-20250609`
    - 充当的是 ci（code interpreter）
- 并发的控制
    - `TokenBucketWorker`: `threading.Semaphore(rate_limit)`

| 层 | 参数 | 当前值 | 作用域 | 对并发的意义 |
|---|---|---:|---|---|
| 训练扇出 | data.train_batch_size | 64 | 每个训练 step | 基础样本数 |
| 训练扇出 | actor_rollout_ref.rollout.n | 16 | 每个样本重复 rollout | 轨迹数放大倍数 |
| AgentLoop 分片 | actor_rollout_ref.rollout.agent.num_workers | 默认 8 | AgentLoop worker 数 | 把轨迹切块并发跑 |
| 单轨迹工具并发 | actor_rollout_ref.rollout.multi_turn.max_parallel_calls | 默认 1 | 同一轮内 tool call | 每条轨迹每轮最多并行几个工具 |
| 工具执行池 | num_workers | 128 | 每个 Tool 实例自己的 ExecutionWorker actor | 该执行池的 max_concurrency |
| 全局限流 | enable_global_rate_limit=true, rate_limit=128 | 128 | 全局 TokenBucket（命名单例 actor） | 集群内同时执行的 tool 请求硬上限 |

```
run_qwen2_7b_dapo.sh
-> verl.trainer.main_ppo
  -> PPO RayTrainer
    -> ActorRollout WorkerGroup
    -> AgentLoopManager (async rollout mode)
      -> RolloutReplicas (vLLMReplica, num_replicas = world_size / rollout_world_size)
        -> vLLMHttpServer actors
        -> vLLMAsyncRollout workers (ZeroMQ backend)
      -> AgentLoopWorker actors (default 8)
        -> ToolAgentLoop (per-sample实例; class级工具初始化)
          -> tools["code_interpreter"] = CustomSandboxFusionTool
            -> execution_pool (Ray actor, max_concurrency=num_workers)
            -> global TokenBucketWorker(name="rate-limiter")
            -> HTTP POST to SandboxFusion /run_code
```

```python
@ray.remote(concurrency_groups={"acquire": 1, "release": 10})
class TokenBucketWorker:
    def __init__(self, rate_limit: int):
        self.rate_limit = rate_limit
        # this only used for observalability
        self.current_count = 0
        self._semaphore = threading.Semaphore(rate_limit)

    @ray.method(concurrency_group="acquire")
    def acquire(self):
        self._semaphore.acquire()
        self.current_count += 1

    @ray.method(concurrency_group="release")
    def release(self):
        self._semaphore.release()
        self.current_count -= 1

    def get_current_count(self):
        return self.current_count
```

- 在 Ray 分布式环境中实现了一个全局的并发限流器（Global Rate Limiter），以防止向后端的 Sandbox Fusion 沙箱服务发送过多请求压垮服务。
- 基于信号量的令牌机制 (Semaphore)
    - 类内部使用了 threading.Semaphore(rate_limit) 作为核心控制器。
        - acquire 方法消耗一个名额（如果名额为 0 则阻塞等待）。
        - release 方法归还一个名额。
    - 这其实并不是传统的漏桶/令牌桶（按时间速率放行），而是一个并发上限控制器（Max Concurrency Limiter），确保同一时刻正在沙箱中执行的代码任务不超过 rate_limit 个。
- misc
    - verl 的架构设计者，非常懂 ray；

1. 训练 step 内先把 batch 扩成 rollout.n 份。见 ray_trainer.py
2. 调 async_rollout_manager.generate_sequences。见 ray_trainer.py
3. Manager wake_up rollout 引擎、按 worker 数切块、下发给每个 AgentLoopWorker。见 agent_loop.py
4. 每个 worker 对 chunk 内每个样本 asyncio.create_task 并发运行 agent loop。见 agent_loop.py
5. 进入 ToolAgentLoop 状态机：PENDING -> GENERATING -> PROCESSING_TOOLS -> ...。见 tool_agent_loop.py
6. GENERATING 调 server manager，从某个 vLLM replica 取 token 输出。见 tool_agent_loop.py
7. format=hermes 时，parser 从 `<tool_call>...</tool_call>` 提取函数调用。见 tool_parser.py
8. 进入 PROCESSING_TOOLS，每轮最多取 max_parallel_calls 个 tool call 并发执行。见 tool_agent_loop.py
9. _call_tool 调 tool.create -> tool.execute -> tool.release。见 tool_agent_loop.py
10. CustomSandboxFusionTool.execute 做代码清洗（提取 fenced python、补最后一行 print），然后投递到执行池。见
 retool.py
11. 执行池内部先 acquire 全局 token，再调用 _process_single_case -> requests.post(/run_code)。见
 sandbox_fusion_tools.py, utils.py（这里的 token 本质上就是 Semaphore 的许可（permit）。）
12. tool 返回文本被追加到消息里，同时这些 observation token 的 response_mask=0。见 tool_agent_loop.py
13. 满足终止条件后回传 Trainer（长度/轮数上限）。见 tool_agent_loop.py
14. 奖励函数 compute_score 读取 num_turns，错误答案时给“多轮/调工具”一点奖励修正。见 retool.py

```mermaid
sequenceDiagram
  autonumber
  participant T as PPO RayTrainer(step k)
  participant M as AgentLoopManager
  participant W as AgentLoopWorker[i]
  participant A as ToolAgentLoop(sample j)
  participant SV as vLLM server
  participant TB as code_interpreter tool
  participant P as ExecutionWorker pool
  participant B as TokenBucket(rate_limit)
  participant SF as SandboxFusion /run_code

  T->>T: gen_batch.repeat(rollout.n)
  T->>M: generate_sequences(gen_batch_output)
  M->>M: wake_up rollout replicas
  M->>W: dispatch chunk_i (parallel i=1..agent.num_workers)
  W->>W: create_task per sample in chunk (parallel)
  W->>A: run state machine

  A->>SV: generate(prompt_ids)
  SV-->>A: response_ids
  A->>A: parse <tool_call> (hermes)

  A->>TB: _call_tool(create->execute->release)
  TB->>P: execute.remote(execute_code)
  P->>B: acquire token
  B-->>P: granted
  P->>SF: HTTP POST /run_code
  SF-->>P: stdout/stderr
  P->>B: release token
  P-->>TB: ToolResponse
  TB-->>A: tool observation
  A->>A: append tool tokens (response_mask=0)
  A->>SV: next generate ... (loop until stop)

  A-->>W: AgentLoopOutput
  W-->>M: DataProto chunk + metrics
  M->>M: concat + sleep replicas
  M-->>T: gen_batch_output
```

- why AgentLoopWorker
    - “大批量轨迹”拆成可并行执行的 worker 分片，避免单事件循环成为瓶颈。
- 并发闸门
    - 样本级并发：每个 worker 对 chunk 内样本并发 task。
    - 轨迹内并发：每轮最多 max_parallel_calls 个工具调用（默认 1）。
    - tool 执行池并发：num_workers（你这里 128）。
    - 全局并发硬上限：rate_limit（你这里 128，全局共享 token bucket）。